# Improved PyG GIN
Early stopping, LR scheduler, dropout, hidden=128, epochs=200.

In [1]:
import pandas as pd
import numpy as np
from rdkit import Chem
import torch
from tqdm import tqdm

from torch_geometric.data import Data, InMemoryDataset
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.nn import GINConv, global_mean_pool

from ogb.utils import smiles2graph as smiles2graph_ogb
from ogb.graphproppred.mol_encoder import AtomEncoder

import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping
from pytorch_lightning.loggers import CSVLogger
import torch.nn.functional as F
from torch import nn

In [2]:
from tdc.single_pred import ADME
data = ADME(name='Lipophilicity_AstraZeneca')
split = data.get_split()

Found local copy...
Loading...
Done!


In [3]:
# Missing values, canonicalization, duplicates
split['train']['_source'] = 'train'
split['valid']['_source'] = 'valid'
split['test']['_source'] = 'test'
combined = pd.concat([split['train'], split['valid'], split['test']], ignore_index=True)

combined['Mol'] = combined.Drug.apply(Chem.MolFromSmiles)
combined = combined.dropna(subset=['Mol'])
combined['Canonical_SMILES'] = combined.Mol.apply(lambda x: Chem.MolToSmiles(x, canonical=True))
combined.drop_duplicates(subset='Canonical_SMILES', keep='first', inplace=True)

split['train'] = combined[combined['_source'] == 'train'].drop(columns=['_source'])
split['valid'] = combined[combined['_source'] == 'valid'].drop(columns=['_source'])
split['test'] = combined[combined['_source'] == 'test'].drop(columns=['_source'])

print(f"Training set shape: {split['train'].shape}")
print(f"Validation set shape: {split['valid'].shape}")
print(f"Test set shape: {split['test'].shape}")

Training set shape: (2940, 5)
Validation set shape: (420, 5)
Test set shape: (840, 5)


In [4]:
TARGET_COL = 'Y'
SMILES_COL = 'Canonical_SMILES'
print(f"Target range: [{combined[TARGET_COL].min():.2f}, {combined[TARGET_COL].max():.2f}] log(mol/L)")

Target range: [-1.50, 4.50] log(mol/L)


In [5]:
class LipophilicityGraphData(InMemoryDataset):
    def __init__(self, root, df, split_name, smiles_col='Canonical_SMILES', target_col='Y'):
        self.df = df[[smiles_col, target_col]].reset_index(drop=True).copy()
        self.smiles_col = smiles_col
        self.target_col = target_col
        self.split_name = split_name
        super().__init__(root)
        self.data, self.slices = torch.load(self.processed_paths[0], weights_only=False)

    @property
    def raw_file_names(self):
        return []

    @property
    def processed_file_names(self):
        return [f'{self.split_name}.pt']

    def download(self):
        pass

    def process(self):
        data_list = []
        for _, row in tqdm(self.df.iterrows(), total=len(self.df), desc=f'Featurising {self.split_name}'):
            smi = row[self.smiles_col]
            y_val = float(row[self.target_col])
            g = smiles2graph_ogb(smi)
            data_list.append(
                Data(
                    x=torch.tensor(g['node_feat'], dtype=torch.long),
                    edge_index=torch.tensor(g['edge_index'], dtype=torch.long),
                    edge_attr=torch.tensor(g['edge_feat'], dtype=torch.long),
                    y=torch.tensor([y_val], dtype=torch.float),
                )
            )
        data, slices = self.collate(data_list)
        torch.save((data, slices), self.processed_paths[0])

train_pyg = LipophilicityGraphData('./lipophilicity_pyg_ogb', split['train'], 'train')
valid_pyg = LipophilicityGraphData('./lipophilicity_pyg_ogb', split['valid'], 'valid')
test_pyg = LipophilicityGraphData('./lipophilicity_pyg_ogb', split['test'], 'test')

train_pyg_loader = PyGDataLoader(train_pyg, batch_size=64, shuffle=True)
val_pyg_loader = PyGDataLoader(valid_pyg, batch_size=64, shuffle=False)
test_pyg_loader = PyGDataLoader(test_pyg, batch_size=64, shuffle=False)

print(f'Train: {len(train_pyg)}  Val: {len(valid_pyg)}  Test: {len(test_pyg)}')

Train: 2940  Val: 420  Test: 840


In [6]:
class LightningGIN(pl.LightningModule):
    def __init__(self, hidden=128, lr=1e-3, dropout=0.2):
        super().__init__()
        self.save_hyperparameters()
        self.atom_emb = AtomEncoder(emb_dim=hidden)
        self.mlp1 = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, hidden))
        self.mlp2 = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, hidden))
        self.conv1 = GINConv(self.mlp1)
        self.conv2 = GINConv(self.mlp2)
        self.dropout = dropout
        self.readout = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, x, edge_index, batch):
        x = self.atom_emb(x)
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = global_mean_pool(x, batch)
        return self.readout(x)

    def _step(self, batch, stage):
        y_hat = self(batch.x, batch.edge_index, batch.batch).squeeze(-1)
        y = batch.y.squeeze(-1)
        loss = F.mse_loss(y_hat, y)
        self.log(f'{stage}/rmse', loss.sqrt(), batch_size=batch.num_graphs, prog_bar=True, on_epoch=True)
        return loss

    def training_step(self, batch, _):
        return self._step(batch, 'train')

    def validation_step(self, batch, _):
        self._step(batch, 'val')

    def test_step(self, batch, _):
        self._step(batch, 'test')

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.lr, weight_decay=1e-5)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'monitor': 'val/rmse'
            }
        }

gin_lightning = LightningGIN(hidden=128, lr=1e-3, dropout=0.2)

In [7]:
logger = CSVLogger('logs', name='gin_experiment')
early_stop = EarlyStopping(monitor='val/rmse', mode='min', patience=20)

trainer = pl.Trainer(
    max_epochs=200,
    logger=logger,
    enable_progress_bar=True,
    log_every_n_steps=5,
    enable_checkpointing=False,
    enable_model_summary=False,
    callbacks=[early_stop],
)
trainer.fit(gin_lightning, train_pyg_loader, val_pyg_loader)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

... (136 outputs left out)

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

In [8]:
# Test evaluation
_ = trainer.test(gin_lightning, test_pyg_loader, verbose=True)

gin_lightning.eval()
y_true_pyg, y_pred_pyg = [], []
with torch.no_grad():
    for batch in test_pyg_loader:
        y_hat = gin_lightning(batch.x, batch.edge_index, batch.batch).squeeze(-1)
        y_true_pyg.extend(batch.y.squeeze(-1).tolist())
        y_pred_pyg.extend(y_hat.tolist())

y_true_pyg = np.array(y_true_pyg)
y_pred_pyg = np.array(y_pred_pyg)
rmse_pyg = np.sqrt(np.mean((y_pred_pyg - y_true_pyg) ** 2))
r2_pyg = 1 - np.sum((y_true_pyg - y_pred_pyg) ** 2) / np.sum((y_true_pyg - y_true_pyg.mean()) ** 2)

print(f'PyG GIN — Test RMSE: {rmse_pyg:.4f}  R2: {r2_pyg:.4f}')

Testing: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test/rmse           0.6614843010902405
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
PyG GIN — Test RMSE: 0.6766  R2: 0.6902
